In [1]:
import numpy as np
import pandas as pd


SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1


def load_ratings(path="ratings.dat"):
    df = pd.read_csv(
        path,
        sep="::",
        engine="python",
        names=["user_id", "movie_id", "rating", "timestamp"],
        encoding="latin-1",
    )
    return df


def make_split(df, seed=SEED):
    rng = np.random.default_rng(seed)

    # shuffle all rows first
    idx = np.arange(len(df))
    rng.shuffle(idx)
    df = df.iloc[idx].reset_index(drop=True)

    n = len(df)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    train_df = df.iloc[:n_train].copy()
    val_df = df.iloc[n_train:n_train + n_val].copy()
    test_df = df.iloc[n_train + n_val:].copy()

    # ensure every user/movie in val/test also appears in train
    train_users = set(train_df["user_id"].unique())
    train_movies = set(train_df["movie_id"].unique())

    def move_unseen_rows(src_df, train_df, train_users, train_movies):
        keep_rows = []
        move_rows = []

        for _, row in src_df.iterrows():
            user_ok = row["user_id"] in train_users
            movie_ok = row["movie_id"] in train_movies

            if user_ok and movie_ok:
                keep_rows.append(row)
            else:
                move_rows.append(row)
                train_users.add(row["user_id"])
                train_movies.add(row["movie_id"])

        if move_rows:
            move_df = pd.DataFrame(move_rows)
            train_df = pd.concat([train_df, move_df], ignore_index=True)

        if keep_rows:
            new_src_df = pd.DataFrame(keep_rows)
        else:
            new_src_df = pd.DataFrame(columns=src_df.columns)

        return new_src_df.reset_index(drop=True), train_df.reset_index(drop=True), train_users, train_movies

    val_df, train_df, train_users, train_movies = move_unseen_rows(
        val_df, train_df, train_users, train_movies
    )
    test_df, train_df, train_users, train_movies = move_unseen_rows(
        test_df, train_df, train_users, train_movies
    )

    return train_df, val_df, test_df


def main():
    df = load_ratings("ratings.dat")
    train_df, val_df, test_df = make_split(df, seed=SEED)

    train_df.to_csv("train_ratings.csv", index=False)
    val_df.to_csv("val_ratings.csv", index=False)
    test_df.to_csv("test_ratings.csv", index=False)

    print("Saved:")
    print(f"train: {len(train_df)}")
    print(f"val:   {len(val_df)}")
    print(f"test:  {len(test_df)}")


if __name__ == "__main__":
    main()

Saved:
train: 800204
val:   100002
test:  100003
